In [2]:
from pathlib import Path

chroma_dir = Path("data/processed/chroma_final")

for path in chroma_dir.rglob("*"):
    if path.is_file():
        print(f"{path.relative_to(chroma_dir)} | {path.stat().st_size:,} bytes")

In [3]:
from pathlib import Path

print("Diretório atual:", Path.cwd())

chroma_dir = Path(
    r"C:\Users\SAMSUNG\Desktop\Pós-graduação\tech-challenge-deeplearning\data\processed\chroma_final"
)

print("Pasta existe:", chroma_dir.exists())

if chroma_dir.exists():
    arquivos = list(chroma_dir.rglob("*"))
    print("Quantidade de itens:", len(arquivos))

    for path in arquivos:
        if path.is_file():
            print(
                f"{path.relative_to(chroma_dir)} | "
                f"{path.stat().st_size:,} bytes"
            )

Diretório atual: c:\Users\SAMSUNG\Desktop\Pós-graduação\tech-challenge-deeplearning\notebooks
Pasta existe: True
Quantidade de itens: 3
chroma.sqlite3 | 66,715,648 bytes
42f2e314-8135-4556-9b55-45de2ca38a5d\index_metadata.pickle | 3,767,688 bytes


In [4]:

from pathlib import Path
import tempfile

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings


def main():
    embeddings = HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-base",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

    # Usa um diretório temporário novo para o teste.
    with tempfile.TemporaryDirectory() as temp_dir:
        documents = [
            Document(
                page_content="passage: Atraso na entrega",
                metadata={"review_id": "teste-1"},
            ),
            Document(
                page_content="passage: Produto chegou corretamente",
                metadata={"review_id": "teste-2"},
            ),
        ]

        store = Chroma.from_documents(
            documents=documents,
            embedding=embeddings,
            collection_name="teste_hnsw",
            persist_directory=temp_dir,
        )

        print("Registros após criação:", store._collection.count())

        del store

        # Reabre o índice em uma nova instância.
        reopened = Chroma(
            collection_name="teste_hnsw",
            persist_directory=temp_dir,
            embedding_function=embeddings,
        )

        print("Registros após reabertura:", reopened._collection.count())

        results = reopened.similarity_search(
            "query: atraso na entrega",
            k=1,
        )

        print("Documentos recuperados:", len(results))


if __name__ == "__main__":
    main()


c:\Users\SAMSUNG\Desktop\Pós-graduação\tech-challenge-deeplearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 356.15it/s]


Registros após criação: 2
Registros após reabertura: 2
Documentos recuperados: 1


PermissionError: [WinError 32] O arquivo já está sendo usado por outro processo: 'C:\\Users\\SAMSUNG\\AppData\\Local\\Temp\\tmpqxw6s9lz\\50ef163c-91fd-4f14-b7cd-0b85d9db04f1\\data_level0.bin'